### Open in Colab 
[![Open in Colab](https://upload.wikimedia.org/wikipedia/commons/thumb/d/d0/Google_Colaboratory_SVG_Logo.svg/56px-Google_Colaboratory_SVG_Logo.svg.png)](https://colab.research.google.com/github/SimpNick6703/Data-Extraction-and-Analysis/blob/main/Extract.ipynb) 

Importing the files from GitHub for use in Colab

In [ ]:
# Cell to be un-commented if opened in Colab
# !git clone https://github.com/SimpNick6703/Data-Extraction-and-Analysis.git
# !cd Data-Extraction-and-Analysis

For installing requirements if not present

In [25]:
# Cell to be un-commented if opened in Colab
# # !pip install pandas BeautifulSoup

This program fulfils the objective of data extraction from a given file `Input.xlsx` and text analysis based on the rules provided in `Text Analysis.docx`. After text analysis, the result is saved inside a new file `Text Analysis Results.xlsx` of which the format is provided as `Output Data Structure.xlsx`. Another file `Output.xlsx` which is copy of `Input.xlsx` is used as base for saving the result and avoiding memory errors with `Input.xlsx`.

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import os

Function for defining the input files and error handling

In [ ]:
input_file = 'Data-Extraction-and-Analysis/Input.xlsx'
def read_file(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            return file.read().splitlines()
    except UnicodeDecodeError:
        try:
            with open(file_path, 'r', encoding='latin-1') as file:
                return file.read().splitlines()
        except UnicodeDecodeError:
            with open(file_path, 'r', encoding='cp1252') as file:
                return file.read().splitlines()

Function for extraction of `paragraph`, `Heading 1` and `List Items` from the URLs in Input file in a proper format. It is noticed that the articles are inside `td-post-content tagdiv-type` under `div` element in the URLs. This can be seen by using `View Page Source` or `Inspect Element` in a browser.

In [ ]:
def extract_article(soup):
    title_tag = soup.find('title')
    title = title_tag.get_text(strip=True) if title_tag else 'No title found'
    
    article_tag = soup.find('div', class_='td-post-content tagdiv-type')
    
    if article_tag:
        elements = article_tag.find_all(['p', 'h1', 'li'])
    else:
        elements = []
    
    article_text = '\n'.join([element.get_text(strip=True) for element in elements])
    return title, article_text


Function for saving the extracted data from URLs to `URL_ID.txt` files under a folder `articles`. A new folder will be created if one doesn't exist.

In [ ]:
def save_article_to_file(url_id, title, article_text, output_folder='articles'):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    file_path = os.path.join(output_folder, f'{url_id}.txt')
    with open(file_path, 'w', encoding='utf-8') as file:
        file.write(f"{title}\n\n{article_text}")

Following function is used for calling the above functions but is Single Threaded in nature. It is slower than Multi-Thread function (following this) but more stable and can be used to when multithreading is not available. 

By default, multi-threaded function is used here and single threaded function is commented out.

In [ ]:
# Comment out this cell if wish to run multi-thread (commented out by default)

# df = pd.read_excel(input_file)
# for index, row in df.iterrows():
#     url_id = row['URL_ID']
#     url = row['URL']
    
#     response = requests.get(url)
#     if response.status_code == 200:
#         soup = BeautifulSoup(response.content, 'html.parser')
#         title, article_text = extract_article(soup)
#         save_article_to_file(url_id, title, article_text)
#     else:
#         print(f"Failed to fetch the content from URL: {url}")

In [ ]:
# Comment out this cell if wish to run single thread mode

from concurrent.futures import ThreadPoolExecutor, as_completed

Function for multi-threaded data extraction.

In [ ]:
# Comment out this cell if wish to run single thread mode

def fetch_and_save_article(row):
    url_id = row['URL_ID']
    url = row['URL']
    try:
        response = requests.get(url)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')
            title, article_text = extract_article(soup)
            save_article_to_file(url_id, title, article_text)
        else:
            print(f"Failed to fetch the content from URL: {url}")
    except Exception as e:
        print(f"Exception for URL {url}: {e}")

Function for calling multithreaded data extraction function.

In [ ]:
df = pd.read_excel(input_file)

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = [executor.submit(fetch_and_save_article, row) for index, row in df.iterrows()]
    for future in as_completed(futures):
        future.result()

Installation of `nltk` and `syllapy` for text analysis if not installed.

In [ ]:
# Cell to be un-commented if opened in Colab

# !pip install nltk syllapy

In [ ]:
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
import re
import syllapy

Ensuring the dependencies are available.

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')

Defining the `Stopwords` and `MasterDictionary` folders containing various files required for text analysis, given for the Text Analysis rules.

In [ ]:
stopwords_folder = 'Data-Extraction-and-Analysis/StopWords'
master_dict_folder = 'Data-Extraction-and-Analysis/MasterDictionary'

Reading the files in defined folders.

In [ ]:
stop_words = set()
for filename in os.listdir(stopwords_folder):
    if filename.endswith(".txt"):
        stop_words.update(read_file(os.path.join(stopwords_folder, filename)))

positive_words = set(read_file(os.path.join(master_dict_folder, 'positive-words.txt')))
negative_words = set(read_file(os.path.join(master_dict_folder, 'negative-words.txt')))

Function to define rules for Text Analysis as per instructions given in the `Text Analysis.docx`.

In [ ]:
def analyze_text(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        text = file.read()
    
    words = word_tokenize(text)
    sentences = sent_tokenize(text)
    
    words_cleaned = [word.lower() for word in words if word.isalpha() and word.lower() not in stop_words]
    
    positive_score = sum(1 for word in words_cleaned if word in positive_words)
    negative_score = sum(1 for word in words_cleaned if word in negative_words)
    
    polarity_score = (positive_score - negative_score) / ((positive_score + negative_score) + 0.000001)
    subjectivity_score = (positive_score + negative_score) / (len(words_cleaned) + 0.000001)
    
    avg_sentence_length = len(words_cleaned) / len(sentences)
    
    complex_words = [word for word in words_cleaned if syllapy.count(word) > 2]
    percentage_complex_words = len(complex_words) / len(words_cleaned)
    
    fog_index = 0.4 * (avg_sentence_length + percentage_complex_words)
    
    avg_words_per_sentence = len(words_cleaned) / len(sentences)
    
    word_count = len(words_cleaned)
    
    syllable_count = sum(syllapy.count(word) for word in words_cleaned)
    avg_syllable_per_word = syllable_count / word_count
    
    personal_pronouns = len(re.findall(r'\b(I|we|my|ours|us)\b', text, re.I))
    
    avg_word_length = sum(len(word) for word in words_cleaned) / word_count
    
    return {
        'Positive Score': positive_score,
        'Negative Score': negative_score,
        'Polarity Score': polarity_score,
        'Subjectivity Score': subjectivity_score,
        'Avg Sentence Length': avg_sentence_length,
        'Percentage of Complex Words': percentage_complex_words,
        'Fog Index': fog_index,
        'Avg Number of Words Per Sentence': avg_words_per_sentence,
        'Complex Word Count': len(complex_words),
        'Word Count': word_count,
        'Syllable Per Word': avg_syllable_per_word,
        'Personal Pronouns': personal_pronouns,
        'Avg Word Length': avg_word_length,
    }

Defining the Output Template (file contents are same as `Input.xlsx` as the program will be making new columns in the results file itself).

In [ ]:
output_template_file = 'Data-Extraction-and-Analysis/Output.xlsx'

Reading the output template and calling text analysis function.

In [ ]:
output_template_df = pd.read_excel(output_template_file)
results = []

articles_folder = 'articles'
for filename in os.listdir(articles_folder):
    if filename.endswith(".txt"):
        url_id = filename.split('.')[0]
        file_path = os.path.join(articles_folder, filename)
        analysis = analyze_text(file_path)
        analysis['URL_ID'] = str(url_id)
        results.append(analysis)

Converting the `results` obtained to dataframes.

In [ ]:
results_df = pd.DataFrame(results)

Getting the results dataframes ready for exporting by converting to output dataframes.

In [ ]:
output_df = output_template_df.merge(results_df, on='URL_ID', how='left')

Exporting the dataframes to `Text Analysis Results.xlsx`.

In [ ]:
output_df.to_excel('Data-Extraction-and-Analysis/Text Analysis Results.xlsx', index=False)